In [27]:
%pip install joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import pandas as pd

file_path = "../data/processed/heart_disease_uci_processed.csv"
df = pd.read_csv(file_path)

In [29]:
# Criar features adicionais sobre o dataset já pré-processado (Aula 2)
df_engineered = df.copy()

# Garantir presença do alvo
y = df_engineered['target']
X = df_engineered.drop(columns=['target'])

In [30]:
# Classificar a coluna 'age' em categorias
bins = [0, 40, 55, 65, 80]  # Definir os intervalos de idade
labels=['young', 'mid_age', 'senior', 'elderly']

df['age_category'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)
df = pd.get_dummies(df, columns=['age_category'], drop_first=True)
# Exibir as primeiras linhas para verificar
df.head()

,age,trestbps,chol,fbs,thalch,exang,oldpeak,ca,target,sex_Male,...,cp_typical angina,restecg_normal,restecg_st-t abnormality,slope_flat,slope_upsloping,thal_normal,thal_reversable defect,age_category_mid_age,age_category_senior,age_category_elderly
0,63,145.0,233.0,True,150.0,False,2.3,0.0,False,True,...,True,False,False,False,False,False,False,False,True,False
1,67,160.0,286.0,False,108.0,True,1.5,3.0,True,True,...,False,False,False,True,False,True,False,False,False,True
2,67,120.0,229.0,False,129.0,True,2.6,2.0,True,True,...,False,False,False,True,False,False,True,False,False,True
3,37,130.0,250.0,False,187.0,False,3.5,0.0,False,True,...,False,True,False,False,False,True,False,False,False,False
4,41,130.0,204.0,False,172.0,False,1.4,0.0,False,False,...,False,False,False,False,True,True,False,True,False,False


In [31]:
# Criar novas features para melhorar o modelo de classificação

# 2. Razão colesterol/idade
df['colesterol_idade'] = df['chol'] / df['age']

# 3. Pressão arterial normalizada
mean_trestbps = df['trestbps'].mean()
df['pressao_normalizada'] = df['trestbps'] / mean_trestbps

# azão pressão/colesterol: pode sinalizar perfil de risco vascular relativo
df['bp_chol_ratio'] = df['trestbps'] / (df['chol'] + 1)

# 4. Interações entre variáveis
df['idade_colesterol'] = df['age'] * df['chol']

# Exibir as primeiras linhas do dataframe atualizado
df.head()

,age,trestbps,chol,fbs,thalch,exang,oldpeak,ca,target,sex_Male,...,slope_upsloping,thal_normal,thal_reversable defect,age_category_mid_age,age_category_senior,age_category_elderly,colesterol_idade,pressao_normalizada,bp_chol_ratio,idade_colesterol
0,63,145.0,233.0,True,150.0,False,2.3,0.0,False,True,...,False,False,False,False,True,False,3.698413,1.098521,0.619658,14679.0
1,67,160.0,286.0,False,108.0,True,1.5,3.0,True,True,...,False,True,False,False,False,True,4.268657,1.212161,0.557491,19162.0
2,67,120.0,229.0,False,129.0,True,2.6,2.0,True,True,...,False,False,True,False,False,True,3.417910,0.909121,0.521739,15343.0
3,37,130.0,250.0,False,187.0,False,3.5,0.0,False,True,...,False,True,False,False,False,False,6.756757,0.984881,0.517928,9250.0
4,41,130.0,204.0,False,172.0,False,1.4,0.0,False,False,...,True,True,False,True,False,False,4.975610,0.984881,0.634146,8364.0


Buscar Features mais relevanes

In [32]:
df.drop(columns=['age'], inplace=True)

In [33]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.model_selection import train_test_split
# Dividir dados (dataset já numérico, sem OHE adicional)
y = df['target']
X = df.drop(columns=['target'])
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Seleção de features usando ANOVA F-value
selector_preview = SelectKBest(f_classif, k=min(25, X_train.shape[1]))
X_train_selected = selector_preview.fit_transform(X_train, y_train)

X_test_selected = selector_preview.transform(X_test)

selected_mask = selector_preview.get_support()
selected_features = [name for name, keep in zip(feature_names, selected_mask) if keep]
print(f"\nFeatures selecionadas ({len(selected_features)}):")
for i, feat in enumerate(selected_features, 1):
    print(f"{i}. {feat}")


Features selecionadas (24):
1. trestbps
2. chol
3. fbs
4. thalch
5. exang
6. oldpeak
7. ca
8. sex_Male
9. cp_atypical angina
10. cp_non-anginal
11. cp_typical angina
12. restecg_normal
13. restecg_st-t abnormality
14. slope_flat
15. slope_upsloping
16. thal_normal
17. thal_reversable defect
18. age_category_mid_age
19. age_category_senior
20. age_category_elderly
21. colesterol_idade
22. pressao_normalizada
23. bp_chol_ratio
24. idade_colesterol


In [34]:
import numpy as np

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, precision_score, recall_score, f1_score

# Comparação inicial entre diferentes algoritmos de classificação
model_configs = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Support Vector Machine": SVC(kernel="rbf", probability=True, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

trained_models = {}
results = []

for model_name, estimator in model_configs.items():
    steps = []
    if model_name == "Support Vector Machine":
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", estimator))
    pipeline = Pipeline(steps)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = np.nan
    if hasattr(pipeline, "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    elif hasattr(pipeline, "decision_function"):
        y_scores = pipeline.decision_function(X_test)
        roc_auc = roc_auc_score(y_test, y_scores)

    print(f"=== {model_name} ===")
    print(f"Acurácia no conjunto de teste: {accuracy:.3f}")
    print(classification_report(y_test, y_pred, target_names=["Sem doença", "Com doença"]))
    print("-" * 70)

    trained_models[model_name] = pipeline
    results.append({
        "Modelo": model_name,
        "Acurácia": accuracy,
        "Precisão": precision,
        "Recall": recall,
        "F1": f1,
        "ROC AUC": roc_auc,
    })

results_df = pd.DataFrame(results)
results_df.sort_values(by="Acurácia", ascending=False).reset_index(drop=True).round(3)

=== Decision Tree ===
Acurácia no conjunto de teste: 0.788
              precision    recall  f1-score   support

  Sem doença       0.77      0.74      0.76        82
  Com doença       0.80      0.82      0.81       102

    accuracy                           0.79       184
   macro avg       0.79      0.78      0.78       184
weighted avg       0.79      0.79      0.79       184

----------------------------------------------------------------------
=== Random Forest ===
Acurácia no conjunto de teste: 0.842
              precision    recall  f1-score   support

  Sem doença       0.85      0.78      0.82        82
  Com doença       0.83      0.89      0.86       102

    accuracy                           0.84       184
   macro avg       0.84      0.84      0.84       184
weighted avg       0.84      0.84      0.84       184

----------------------------------------------------------------------
=== Support Vector Machine ===
Acurácia no conjunto de teste: 0.864
              prec

,Modelo,Acurácia,Precisão,Recall,F1,ROC AUC
0,Support Vector Machine,0.864,0.853,0.912,0.882,0.925
1,Random Forest,0.842,0.835,0.892,0.863,0.909
2,Gradient Boosting,0.842,0.835,0.892,0.863,0.907
3,Decision Tree,0.788,0.800,0.824,0.812,0.784


In [36]:
import joblib
import os

best_model = trained_models["Support Vector Machine"]

# Salvar o melhor modelo SVM treinado
model_path = '../models/best_svm_model.pkl'
joblib.dump(best_model, model_path)

print(f"Modelo SVM salvo com sucesso em: {model_path}")
print(f"\nDetalhes do modelo exportado:")
print(f"Acurácia: {accuracy:.3f}")
print(f"ROC AUC: {roc_auc:.3f}")

Modelo SVM salvo com sucesso em: ../models/best_svm_model.pkl

Detalhes do modelo exportado:
Acurácia: 0.842
ROC AUC: 0.907


In [26]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

# Definir o pipeline para o SVC
svc_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(probability=True, random_state=42))
])

# Definir o espaço de busca para os hiperparâmetros
param_distributions = {
    "svc__C": uniform(0.1, 200),
    "svc__gamma": uniform(0.001, 1),
    "svc__kernel": ["rbf"],
    "svc__degree": [2, 3, 4, 5, 6],
    "svc__coef0": uniform(0, 1)
}

# Configurar o RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=svc_pipeline,
    param_distributions=param_distributions,
    n_iter=50,  # Número de combinações a serem testadas
    scoring="accuracy",
    cv=5,  # Validação cruzada com 5 folds
    random_state=42,
    n_jobs=-1  # Usar todos os núcleos disponíveis
)

# Executar o RandomizedSearchCV
random_search.fit(X_train, y_train)

# Exibir os melhores hiperparâmetros e a acurácia correspondente
print("Melhores hiperparâmetros:")
print(random_search.best_params_)
print(f"Melhor acurácia obtida: {random_search.best_score_:.3f}")

# Avaliar o modelo no conjunto de teste
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)
print("\nDesempenho no conjunto de teste:")
print(classification_report(y_test, y_pred, target_names=["Sem doença", "Com doença"]))

Melhores hiperparâmetros:
{'svc__C': np.float64(22.278164162366267), 'svc__coef0': np.float64(0.4393365018657701), 'svc__degree': 4, 'svc__gamma': np.float64(0.03242918568673425), 'svc__kernel': 'rbf'}
Melhor acurácia obtida: 0.810

Desempenho no conjunto de teste:
              precision    recall  f1-score   support

  Sem doença       0.79      0.77      0.78        82
  Com doença       0.82      0.83      0.83       102

    accuracy                           0.80       184
   macro avg       0.80      0.80      0.80       184
weighted avg       0.80      0.80      0.80       184

